# Numerical Integration Error Comparison
Compare Trapezoidal, Simpson, and Gauss-Legendre quadrature on 
$\int_0^1 e^x \, dx = e - 1$.

In [2]:
import numpy as np

def trapezoidal_integrate(f, a, b, n):
    """Composite trapezoidal rule with n subintervals."""
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h * (0.5 * y[0] + np.sum(y[1:-1]) + 0.5 * y[-1])

def simpson_integrate(f, a, b, n):
    """Composite Simpson rule with n subintervals (n must be even)."""
    if n % 2 != 0:
        raise ValueError('Simpson rule requires even n.')
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return (h / 3) * (y[0] + y[-1] + 4 * np.sum(y[1:-1:2]) + 2 * np.sum(y[2:-2:2]))

def gauss_legendre_integrate(f, a, b, n):
    """n-point Gauss-Legendre quadrature on [a, b]."""
    xi, wi = np.polynomial.legendre.leggauss(n)
    x = 0.5 * (b - a) * xi + 0.5 * (a + b)
    return 0.5 * (b - a) * np.sum(wi * f(x))

In [3]:
# Test function and exact value
f = np.exp
a, b = 0.0, 1.0
exact = np.e - 1

print(f'Exact value: {exact:.15f}')
print('-' * 88)
print(f"{'N':>4} | {'Trap Value':>15} | {'Trap Err':>11} | {'Simpson Err':>11} | {'Gauss Err':>11}")
print('-' * 88)

# N is subinterval count for Trap/Simpson and point count for Gauss
for N in [2, 4, 8, 16, 32]:
    trap = trapezoidal_integrate(f, a, b, N)
    simp = simpson_integrate(f, a, b, N)
    gauss = gauss_legendre_integrate(f, a, b, N)

    err_trap = abs(trap - exact)
    err_simp = abs(simp - exact)
    err_gauss = abs(gauss - exact)

    print(f'{N:4d} | {trap:15.12f} | {err_trap:11.3e} | {err_simp:11.3e} | {err_gauss:11.3e}')

Exact value: 1.718281828459045
----------------------------------------------------------------------------------------
   N |      Trap Value |    Trap Err | Simpson Err |   Gauss Err
----------------------------------------------------------------------------------------
   2 |  1.753931092465 |   3.565e-02 |   5.793e-04 |   3.855e-04
   4 |  1.727221904558 |   8.940e-03 |   3.701e-05 |   9.330e-10
   8 |  1.720518592164 |   2.237e-03 |   2.326e-06 |   0.000e+00
  16 |  1.718841128580 |   5.593e-04 |   1.456e-07 |   0.000e+00
  32 |  1.718421660316 |   1.398e-04 |   9.103e-09 |   2.220e-16


In [8]:
def f(x):
    return np.sin(x)

a = 0.0
b = np.pi/2

epsrel_max = 1e-10

print('Trapezoidal')

k_max = 20

Tk = np.zeros(k_max+1)

Tk[0] = (f(a) + f(b)) * (b - a) / 2
k = 0
epsrel = 1e30
while epsrel > epsrel_max and k < k_max:
    k += 1
    hk = (b - a) / 2**k
    s = 0.0
    for j in range(1, 2**k, 2):
        s = s + f(a + j * hk)
    s = s * hk
    Tk[k] = 0.5 * Tk[k-1] + s
    epsrel = np.abs(Tk[k] - Tk[k-1]) / np.abs(Tk[k])
    print('k=',k, 'Tk=', Tk[k], 'epsrel=', epsrel)

print('Simpson')

k_max = 20

Sn = np.zeros(k_max+1)
Sn1 = np.zeros(k_max+1)

fa = f(a)
fb = f(b)
Sn1[0] = f((a + b) / 2) 

Sn[0] = Sn1[0] * (b - a)

k = 0
epsrel = 1e30
while k <= k_max and epsrel > epsrel_max:
    k += 1
    n = 2**k
    hn = (b - a) / 2**k
    Sn2 = 0.0
    for j in range(1, n+1):
        Sn2 = Sn2 + f(a + (j - 0.5) * hn)
    Sn[k] = hn / 3 *(fa + fb + 2*Sn1[k-1] + 4*Sn2)
    Sn1[k] = Sn1[k-1] + Sn2
    epsrel = np.abs(Sn[k] - Sn[k-1]) / np.abs(Sn[k])
    print('k=',k, 'Sn=', Sn[k], 'epsrel=', epsrel)


Trapezoidal
k= 1 Tk= 0.9480594489685199 epsrel= 0.1715728752538099
k= 2 Tk= 0.9871158009727754 epsrel= 0.03956612989658006
k= 3 Tk= 0.9967851718861696 epsrel= 0.009700556535263515
k= 4 Tk= 0.9991966804850723 epsrel= 0.0024134473682718956
k= 5 Tk= 0.9997991943200187 epsrel= 0.000602634847446794
k= 6 Tk= 0.9999498000921012 epsrel= 0.0001506133328579172
k= 7 Tk= 0.9999874501175262 epsrel= 3.765049793431978e-05
k= 8 Tk= 0.9999968625352877 epsrel= 9.412447292755246e-06
k= 9 Tk= 0.9999992156341913 epsrel= 2.3531007491987614e-06
k= 10 Tk= 0.9999998039085709 epsrel= 5.882744950092835e-07
k= 11 Tk= 0.9999999509771443 epsrel= 1.4706858064033056e-07
k= 12 Tk= 0.9999999877442864 epsrel= 3.676714255925885e-08
k= 13 Tk= 0.9999999969360728 epsrel= 9.191786360237478e-09
k= 14 Tk= 0.999999999234017 epsrel= 2.297944170043736e-09
k= 15 Tk= 0.9999999998085037 epsrel= 5.744867915814411e-10
k= 16 Tk= 0.9999999999521243 epsrel= 1.4362055989613298e-10
k= 17 Tk= 0.9999999999880265 epsrel= 3.590217012615326e-11